In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve
)

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
SEEDS = [42, 123, 2023, 777, 999]

In [ ]:
VISUALIZE_CLASS_DISTRIBUTION = True
USE_MIXED_PRECISION           = True
USE_LR_SCHEDULER              = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

labeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/eq-5d-200-records.csv")
unlabeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/eq-5d-2000-unique-random.csv")

for df in [labeled, unlabeled]:
    df["model_text"] = df["Title"].astype(str) + " [SEP] " + df["Abstract"].astype(str)

In [ ]:
if VISUALIZE_CLASS_DISTRIBUTION:
    labeled["Label"].value_counts().plot(kind="bar", title="Class Distribution")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
train_df, test_df = train_test_split(
    labeled,
    test_size=0.35,
    stratify=labeled["Label"],
    random_state=42
)

train_sub, val_sub = train_test_split(
    train_df,
    test_size=0.25,
    stratify=train_df["Label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_sub))
print("Test:", len(test_df))

In [ ]:
MODEL_MAP = {
    "bert": "bert-base-uncased",
    "scibert": "allenai/scibert_scivocab_uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "modernbert": "answerdotai/ModernBERT-base",
    "pubmedbert_base": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "biolinkbert_large": "michiyasunaga/BioLinkBERT-large",
    "biolinkbert_base": "michiyasunaga/BioLinkBERT-base"
}
selected_model = "biolinkbert_base"
BASE_MODEL = MODEL_MAP[selected_model]

In [ ]:

MAX_LEN = 256
BATCH_SIZE = 16

MLM_EPOCHS = 15
TEACHER_EPOCHS = 25
STUDENT_EPOCHS = 35

TEACHER_LR = 2e-5
LR_LIST = [1e-5, 2e-5, 3e-5, 5e-5]

UNSUP_WEIGHT = 0.10
CONF_THRESHOLD = 0.90
MAX_PSEUDO_PER_CLASS = 250

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
def encode(df):

    enc = tokenizer(
        df["model_text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    labels = torch.tensor(df["Label"].values)

    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)


def encode_unlabeled(df):

    enc = tokenizer(
        df["model_text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    return TensorDataset(enc["input_ids"], enc["attention_mask"])

In [ ]:
def train_mlm(df_all):

    model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL).to(DEVICE)

    enc = tokenizer(
        df_all["model_text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    dataset = [
        {"input_ids": i, "attention_mask": m}
        for i, m in zip(enc["input_ids"], enc["attention_mask"])
    ]

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=True,
            mlm_probability=0.15
        )
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)

    for epoch in range(MLM_EPOCHS):

        model.train()

        for batch in loader:

            batch = {k:v.to(DEVICE) for k,v in batch.items()}

            loss = model(**batch).loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model.base_model.state_dict()


mlm_state = train_mlm(pd.concat([train_df, unlabeled], ignore_index=True))

In [ ]:
def train_teacher(df):

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2
    ).to(DEVICE)

    model.base_model.load_state_dict(mlm_state, strict=False)

    loader = DataLoader(encode(df), batch_size=BATCH_SIZE, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=TEACHER_LR)

    for epoch in range(TEACHER_EPOCHS):

        model.train()

        for batch in loader:

            batch = [b.to(DEVICE) for b in batch]

            logits = model(
                input_ids=batch[0],
                attention_mask=batch[1]
            ).logits

            loss = F.cross_entropy(logits, batch[2])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model


teacher = train_teacher(train_df)

In [ ]:
def generate_pseudo_with_full_output(teacher):

    loader = DataLoader(
        encode_unlabeled(unlabeled),
        batch_size=BATCH_SIZE
    )

    teacher.eval()

    probs_all = []

    with torch.no_grad():
        for batch in loader:
            batch = [b.to(DEVICE) for b in batch]

            logits = teacher(
                input_ids=batch[0],
                attention_mask=batch[1]
            ).logits

            probs = torch.softmax(logits, dim=1)
            probs_all.append(probs.cpu())

    probs_all = torch.cat(probs_all)

    conf, preds = torch.max(probs_all, dim=1)

    full_df = unlabeled.copy()

    full_df["pred_label"] = preds.numpy()
    full_df["confidence"] = conf.numpy()
    full_df["prob_0"] = probs_all[:, 0].numpy()
    full_df["prob_1"] = probs_all[:, 1].numpy()


    mask = conf >= CONF_THRESHOLD

    probs_selected = probs_all[mask]
    preds_selected = preds[mask]

    balanced_indices = []

    for c in [0, 1]:
        idx = (preds_selected == c).nonzero(as_tuple=True)[0]
        idx = idx[:MAX_PSEUDO_PER_CLASS]
        balanced_indices.extend(idx.tolist())

    balanced_indices = torch.tensor(balanced_indices)

    texts_selected = unlabeled.iloc[mask.numpy()]["model_text"].iloc[
        balanced_indices.numpy()
    ]

    probs_selected = probs_selected[balanced_indices]

    pseudo_df = pd.DataFrame({
        "model_text": texts_selected.values,
        "soft0": probs_selected[:, 0].numpy(),
        "soft1": probs_selected[:, 1].numpy()
    })

    return pseudo_df, full_df

pseudo_df, unlabeled_predictions_df = generate_pseudo_with_full_output(teacher)
print("Pseudo samples:", len(pseudo_df))

unlabeled_predictions_df.to_csv(
    "/content/drive/MyDrive/eq_5d/semi-supervised/biobert_base_multiple_LR/biobert_unlabeled_predictions_with_confidence.csv",
    index=False
)

In [ ]:
class PseudoDataset(torch.utils.data.Dataset):

    def __init__(self, df):

        self.texts = df["model_text"].values
        self.soft = df[["soft0","soft1"]].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        enc = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.soft[idx])
        )

In [ ]:
all_test_preds = []
all_test_gold = None

for seed in SEEDS:
    print("\n=== Seed:", seed, "===")
    set_seed(seed)

    labels = train_sub["Label"].values
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    sup_loader = DataLoader(
        encode(train_sub),
        batch_size=BATCH_SIZE,
        sampler=WeightedRandomSampler(
            sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
    )
    unsup_loader = DataLoader(
        PseudoDataset(pseudo_df),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    best_student = None
    best_val_auc = 0
    best_lr = None
    best_val_probs = None
    best_val_gold = None

    for LR in LR_LIST:
        student = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL, num_labels=2
        ).to(DEVICE)
        student.base_model.load_state_dict(mlm_state, strict=False)
        optimizer = AdamW(student.parameters(), lr=LR)

        if USE_LR_SCHEDULER:
            total_steps = len(sup_loader) * STUDENT_EPOCHS
            warmup_steps = int(0.1 * total_steps)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )

        scaler = torch.cuda.amp.GradScaler() if USE_MIXED_PRECISION else None

        for epoch in range(STUDENT_EPOCHS):
            student.train()
            unsup_iter = iter(unsup_loader)

            for sup_batch in sup_loader:
                sup_batch = [b.to(DEVICE) for b in sup_batch]
                optimizer.zero_grad()

                try:
                    unsup_batch = next(unsup_iter)
                except StopIteration:
                    unsup_iter = iter(unsup_loader)
                    unsup_batch = next(unsup_iter)
                unsup_batch = [b.to(DEVICE) for b in unsup_batch]

                if USE_MIXED_PRECISION:
                    with torch.cuda.amp.autocast():
                        sup_logits = student(
                            input_ids=sup_batch[0],
                            attention_mask=sup_batch[1]
                        ).logits
                        sup_loss = F.cross_entropy(sup_logits, sup_batch[2])
                        student_logits = student(
                            input_ids=unsup_batch[0],
                            attention_mask=unsup_batch[1]
                        ).logits
                        student_log_probs = F.log_softmax(student_logits, dim=1)
                        unsup_loss = F.kl_div(
                            student_log_probs,
                            unsup_batch[2],
                            reduction="batchmean"
                        )
                        loss = sup_loss + UNSUP_WEIGHT * unsup_loss
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    sup_logits = student(
                        input_ids=sup_batch[0],
                        attention_mask=sup_batch[1]
                    ).logits
                    sup_loss = F.cross_entropy(sup_logits, sup_batch[2])
                    student_logits = student(
                        input_ids=unsup_batch[0],
                        attention_mask=unsup_batch[1]
                    ).logits
                    student_log_probs = F.log_softmax(student_logits, dim=1)
                    unsup_loss = F.kl_div(
                        student_log_probs,
                        unsup_batch[2],
                        reduction="batchmean"
                    )
                    loss = sup_loss + UNSUP_WEIGHT * unsup_loss
                    loss.backward()
                    optimizer.step()

                if USE_LR_SCHEDULER:
                    scheduler.step()

        student.eval()
        val_probs = []
        val_gold = []
        with torch.no_grad():
            for batch in DataLoader(encode(val_sub), batch_size=BATCH_SIZE):
                batch = [b.to(DEVICE) for b in batch]
                logits = student(
                    input_ids=batch[0],
                    attention_mask=batch[1]
                ).logits
                prob = torch.softmax(logits, dim=1)
                val_probs.extend(prob[:,1].cpu().numpy())
                val_gold.extend(batch[2].cpu().numpy())

        val_probs = np.array(val_probs)
        val_gold = np.array(val_gold)
        val_auc = roc_auc_score(val_gold, val_probs)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_student = student
            best_lr = LR
            best_val_probs = val_probs
            best_val_gold = val_gold

    student = best_student
    student.eval()
    test_probs = []
    test_gold = []
    with torch.no_grad():
        for batch in DataLoader(encode(test_df), batch_size=BATCH_SIZE):
            batch = [b.to(DEVICE) for b in batch]
            logits = student(
                input_ids=batch[0],
                attention_mask=batch[1]
            ).logits
            prob = torch.softmax(logits, dim=1)
            test_probs.extend(prob[:,1].cpu().numpy())
            test_gold.extend(batch[2].cpu().numpy())

    test_probs = np.array(test_probs)
    test_gold = np.array(test_gold)
    test_preds = (test_probs >= 0.5).astype(int)

    all_test_preds.append(test_preds)
    if all_test_gold is None:
        all_test_gold = test_gold

In [ ]:
def bootstrap_ci(y_true, preds, metric, n=1000):
    scores = []
    N = len(y_true)
    for _ in range(n):
        idx = np.random.choice(N, N, replace=True)
        if metric == accuracy_score:
            scores.append(metric(y_true[idx], preds[idx]))
        else:
            scores.append(metric(y_true[idx], preds[idx], average="weighted"))
    return np.percentile(scores, [2.5, 97.5])

f1_scores = [f1_score(all_test_gold, p, average="weighted") for p in all_test_preds]
acc_scores = [accuracy_score(all_test_gold, p) for p in all_test_preds]

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)
mean_acc = np.mean(acc_scores)
std_acc = np.std(acc_scores)

best_idx = np.argmax(f1_scores)
best_test_preds = all_test_preds[best_idx]
best_seed = SEEDS[best_idx]

ci_f1 = bootstrap_ci(all_test_gold, best_test_preds, f1_score)
ci_acc = bootstrap_ci(all_test_gold, best_test_preds, accuracy_score)

print("\n" + "="*60)
print("FINAL METRICS SUMMARY")
print("="*60)
print(f"Mean F1 ± Std (across seeds): {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Mean Accuracy ± Std (across seeds): {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Best Seed: {best_seed}")
print(f"F1 (best seed) 95% CI: [{ci_f1[0]:.4f}, {ci_f1[1]:.4f}]")
print(f"Accuracy (best seed) 95% CI: [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}]")

In [ ]:
from sklearn.metrics import classification_report

print("\n" + "="*60)
print("CLASSIFICATION REPORT (Best Seed)")
print("="*60)

print(classification_report(all_test_gold, best_test_preds))

In [ ]:
torch.save(best_student.state_dict(), "/content/drive/MyDrive/eq_5d/semi-supervised/biobert_base_multiple_LR/biobert_with_more_metrics_best_semisupervised_model_2.pt")